# Harvest Slot 사과 단일 이미지 DL 학습 파이프라인 - view balanced 버전

이 노트북은 Kaggle Notebook에서 **사과 데이터셋만 연결한 상태**로 바로 실행할 수 있도록 구성한 사과 전용 학습 문서입니다.

실서비스 입력은 사진 1장입니다. 모델은 먼저 입력 이미지가 `top`, `middle`, `side` 중 어디에 가까운지 판단하고, 그 결과에 맞는 등급 모델을 선택해 A/B/C 등급과 신선도 보조 점수를 계산합니다.

여기서 `middle`은 사용자가 45도 사진을 찍어야 한다는 의미가 아니라, 모델 내부에서 `top`도 아니고 `side`도 아닌 중간/사선 시점 이미지를 따로 다루기 위한 클래스입니다.

이 버전은 같은 사과 번호 또는 `group_no`를 하나의 묶음으로 보고, 같은 사과의 다른 각도 사진이 train/valid/test에 나뉘지 않도록 분리합니다.

추가로 view 모델 학습에서만 `top/middle/side` train 데이터를 균형 있게 downsampling합니다. 등급 모델은 각 view별 전체 train 데이터를 그대로 사용합니다.


## 전체 흐름

1. Kaggle/로컬 실행 환경과 GPU 확인
2. 사과 원천 이미지와 JSON 라벨링 데이터 자동 탐색
3. 이미지-JSON 매칭 후 manifest 생성
4. `cate3` 기준 등급 라벨 생성
5. `angle_direction` 또는 파일명 기준으로 `top/middle/side` 라벨 생성
6. `group_no` 또는 사과 번호 기준으로 train/valid/test 분리
7. 이미지 품질 지표 계산
8. view 모델 train 데이터만 `top/middle/side` 균형 샘플로 구성
9. `top/middle/side` 확인 모델 학습
10. top 전용 등급 모델 학습
11. middle 전용 등급 모델 학습
12. side 전용 등급 모델 학습
13. 단일 이미지 입력 추론과 FastAPI 응답 준비

최종 산출물:

```text
apple_view_top_middle_side_balanced_resnet18_best.pt
apple_top_grade_resnet18_best.pt
apple_middle_grade_resnet18_best.pt
apple_side_grade_resnet18_best.pt
```

서비스 추론 흐름:

```text
사용자 이미지 1장
-> view 모델로 top/middle/side 자동 판단
-> top이면 top 등급 모델 사용
-> middle이면 middle 등급 모델 사용
-> side이면 side 등급 모델 사용
-> A/B/C 등급과 freshness_score 계산
```


## 1. 실행 환경 확인

Kaggle에서 실행할 때는 우측 설정에서 Accelerator(가속기)를 GPU로 설정합니다.

In [ ]:
from __future__ import annotations

import copy
import json
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageFilter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

IS_KAGGLE = Path('/kaggle').exists()
NOTE_DIR = Path.cwd()
PROJECT_ROOT = NOTE_DIR.parent if NOTE_DIR.name == 'Note' else NOTE_DIR
WORKING_ROOT = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT
KAGGLE_INPUT_ROOT = Path('/kaggle/input')

print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('WORKING_ROOT:', WORKING_ROOT)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 2. 사과 전용 설정

Kaggle dataset에는 사과 데이터만 들어 있다고 가정합니다. 지정 경로를 먼저 확인하고, 없으면 `/kaggle/input` 아래에서 자동 탐색합니다.

In [ ]:
TARGET_FRUIT = 'apple'
FRUIT_DISPLAY_NAME = '사과'

APPLE_VARIETIES = ['부사', '양광', 'fuji', 'yanggwang']
APPLE_KEYWORDS = ['apple', '사과', '부사', '양광', 'fuji', 'yanggwang']
SELECTED_VARIETIES: list[str] = []

GRADE_LABELS = ['A', 'B', 'C']
ANGLE_LABELS = ['top', 'middle', 'side']
ANGLE_DETAIL_LABELS = ['top', 'front45', 'front90', 'diagonal45', 'diagonal90']

QC_FOLDER_LABEL_MAP = {'L': 'A', 'M': 'B', 'S': 'C'}
QC_CATE3_LABEL_MAP = {'특': 'A', '상': 'B', '보통': 'C', 'L': 'A', 'M': 'B', 'S': 'C', 'A': 'A', 'B': 'B', 'C': 'C'}
GRADE_SCORE_MAP = {'A': 90.0, 'B': 70.0, 'C': 45.0}

DEFAULT_IMAGE_SIZE = 224
DEFAULT_MODEL_NAME = 'resnet18'
RANDOM_SEED = 42

COMBINE_OFFICIAL_SPLITS_BEFORE_SPLIT = True
TEST_RATIO = 0.3
VALID_RATIO_IN_TRAIN_VALID = 0.2

# 실제 서비스에는 JSON bbox가 없으므로 기본값은 전체 이미지 학습입니다.
USE_BBOX_CROP = False
BBOX_PADDING_RATIO = 0.04

USE_PRETRAINED = True
RUN_ANGLE_TRAINING = True
RUN_TOP_GRADE_TRAINING = True
RUN_MIDDLE_GRADE_TRAINING = True
RUN_SIDE_GRADE_TRAINING = True
USE_WEIGHTED_SAMPLER = False

# train dataset에만 적용합니다. valid/test는 원본 분포를 유지합니다.
# group_id를 우선 순회해 같은 사과의 사진이 과하게 몰리지 않도록 샘플링합니다.
USE_TRAIN_BALANCED_DATASET = True
TRAIN_BALANCE_TARGET = 'min'  # 'min', 'max', 또는 정수. 'min'은 중복 없이 가장 보수적으로 맞춥니다.
TRAIN_BALANCE_GROUP_AWARE = True
TRAIN_BALANCE_MAX_IMAGES_PER_GROUP = 1  # 1이면 가능한 한 서로 다른 사과를 먼저 사용합니다.
TRAIN_BALANCE_RANDOM_SEED = RANDOM_SEED

# 데이터 불균형 보정 설정입니다.
# angle 모델은 시점 x 등급 x 품종, grade 모델은 등급 x 품종 조합 기준으로 train 데이터를 맞춥니다.
BALANCE_GROUP_COLUMNS = {
    'angle': ['angle_label', 'grade_label', 'variety'],
    'top_grade': ['grade_label', 'variety'],
    'middle_grade': ['grade_label', 'variety'],
    'side_grade': ['grade_label', 'variety'],
}
BALANCED_SAMPLES_PER_GROUP = 'max'  # 'max' 또는 정수
MAX_OVERSAMPLE_MULTIPLIER = 3.0

# 불균형/어려운 샘플을 더 반영하기 위한 loss 설정입니다.
USE_CLASS_WEIGHTED_LOSS = False
USE_FOCAL_LOSS = False
FOCAL_GAMMA = 1.5
MAX_CLASS_LOSS_WEIGHT = 3.0

TRAIN_CONFIG = {
    'image_size': DEFAULT_IMAGE_SIZE,
    'batch_size': 64,
    'epochs': 10,
    'patience': 4,
    'learning_rate': 3e-4,
    'weight_decay': 1e-4,
    'num_workers': 2 if IS_KAGGLE else 0,
}

KAGGLE_DATA_ROOT = Path('/kaggle/input/datasets/hmw0320/data-fruits')
LOCAL_DATA_ROOT = PROJECT_ROOT / 'Data' / 'Sample'
MODEL_ROOT = WORKING_ROOT / 'models'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
JSON_EXTS = {'.json'}

print('대상:', TARGET_FRUIT, '/', FRUIT_DISPLAY_NAME)
print('선택 품종:', SELECTED_VARIETIES if SELECTED_VARIETIES else '사과 대상 품종 전체')
print('bbox crop 사용:', USE_BBOX_CROP)
print('모델 저장 경로:', MODEL_ROOT)


## 3. 원천 이미지와 JSON 라벨링 데이터 탐색

원천 이미지를 먼저 찾고, 라벨링 JSON을 별도로 읽어 이미지와 매칭합니다. JSON이 매칭되면 `cate3`, `group_no`, `angle_direction`, `cate2`, `bndbox`를 우선 사용합니다.

In [ ]:
def normalized_text(value: str) -> str:
    return value.lower().replace(' ', '').replace('_', '').replace('-', '').replace('.', '')


def contains_any_keyword(path: Path, keywords: list[str]) -> bool:
    text = normalized_text(str(path))
    return any(normalized_text(keyword) in text for keyword in keywords)


def is_label_data_path(path: Path) -> bool:
    text = str(path).lower()
    return ('라벨' in text) or ('label' in text) or ('annotation' in text)


def is_source_data_path(path: Path) -> bool:
    text = str(path).lower()
    return ('원천' in text) or ('source' in text) or ('raw' in text)


def is_training_path(path: Path) -> bool:
    text = str(path).lower()
    return ('training' in text) or ('train' in text) or ('학습' in text)


def is_validation_path(path: Path) -> bool:
    text = str(path).lower()
    return ('validation' in text) or ('valid' in text) or ('검증' in text)


def official_split_from_path(path: Path) -> str:
    if is_training_path(path):
        return 'training'
    if is_validation_path(path):
        return 'validation'
    return 'unknown'


def candidate_search_roots() -> list[Path]:
    roots = [KAGGLE_DATA_ROOT, KAGGLE_INPUT_ROOT] if IS_KAGGLE else [LOCAL_DATA_ROOT, PROJECT_ROOT / 'Data']
    unique, seen = [], set()
    for root in roots:
        if str(root) not in seen:
            unique.append(root)
            seen.add(str(root))
    return unique


def find_files() -> tuple[list[Path], list[Path]]:
    roots = [root for root in candidate_search_roots() if root.exists()]
    all_images, all_jsons = [], []
    for root in roots:
        all_images.extend(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        all_jsons.extend(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in JSON_EXTS)
    all_images = sorted(set(all_images))
    all_jsons = sorted(set(all_jsons))

    source_images = [p for p in all_images if is_source_data_path(p)]
    all_images = source_images if source_images else [p for p in all_images if not is_label_data_path(p)]
    apple_images = [p for p in all_images if contains_any_keyword(p, APPLE_KEYWORDS)]
    apple_jsons = [p for p in all_jsons if contains_any_keyword(p, APPLE_KEYWORDS) or is_label_data_path(p)]
    return sorted(apple_images), sorted(apple_jsons)


image_files, json_files = find_files()
print('image_files:', len(image_files))
print('json_files:', len(json_files))
print('image 예시:', image_files[:3])
print('json 예시:', json_files[:3])

## 4. JSON 파싱과 이미지 매칭

JSON의 `identifier`와 JSON 파일명을 이용해 원천 이미지와 라벨 정보를 연결합니다.

In [ ]:
def safe_read_json(path: Path) -> dict[str, Any] | None:
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except UnicodeDecodeError:
        return json.loads(path.read_text(encoding='cp949'))
    except Exception as exc:
        print('JSON 읽기 실패:', path, exc)
        return None


def basename_from_identifier(identifier: Any) -> str | None:
    if not identifier:
        return None
    return Path(str(identifier).replace('\\', '/')).name


def build_json_index(json_paths: list[Path]) -> dict[str, dict[str, Any]]:
    index = {}
    for json_path in json_paths:
        data = safe_read_json(json_path)
        if not data:
            continue
        data['_json_path'] = str(json_path)
        keys = {json_path.name, json_path.stem}
        identifier_name = basename_from_identifier(data.get('identifier'))
        if identifier_name:
            keys.add(identifier_name)
            keys.add(Path(identifier_name).stem)
        for key in keys:
            index.setdefault(key, data)
    return index


def match_json_for_image(image_path: Path, json_index: dict[str, dict[str, Any]]) -> dict[str, Any] | None:
    return json_index.get(image_path.name) or json_index.get(image_path.stem)


json_index = build_json_index(json_files)
matched_count = sum(1 for image in image_files if match_json_for_image(image, json_index) is not None)
print('JSON index keys:', len(json_index))
print('matched images:', matched_count, '/', len(image_files))
print('matched ratio:', round(matched_count / len(image_files), 4) if image_files else 0)

## 5. Manifest 생성

Manifest는 학습에 사용할 데이터 표입니다. JSON은 모델 입력이 아니라 정답 라벨, 그룹 분리, 평가 분석을 위한 메타데이터로 사용합니다.

In [ ]:
FILENAME_GROUP_PATTERN = re.compile(r'.*_[LMS]_(\d+)-\d+_')
ANGLE_TOKEN_MAP = {'1TOP': 'top', '2FR45': 'front45', '3FR90': 'front90', '4DI45': 'diagonal45', '5DI90': 'diagonal90'}


def grade_from_json_or_path(data: dict[str, Any] | None, image_path: Path) -> str | None:
    if data:
        cate3 = str(data.get('cate3', '')).strip()
        if cate3 in QC_CATE3_LABEL_MAP:
            return QC_CATE3_LABEL_MAP[cate3]
    for part in reversed(image_path.parts):
        last = part[-1:].upper() if part else ''
        if last in QC_FOLDER_LABEL_MAP:
            return QC_FOLDER_LABEL_MAP[last]
    return None



def angle_detail_from_json_or_name(data: dict[str, Any] | None, image_path: Path) -> str | None:
    if data:
        raw = str(data.get('angle_direction', '')).strip().lower().replace('-', '').replace('_', '')
        angle_map = {
            'top': 'top', '1top': 'top',
            'front45': 'front45', 'fr45': 'front45', '2fr45': 'front45',
            'front90': 'front90', 'fr90': 'front90', '3fr90': 'front90',
            'diagonal45': 'diagonal45', 'diag45': 'diagonal45', 'di45': 'diagonal45', '4di45': 'diagonal45',
            'diagonal90': 'diagonal90', 'diag90': 'diagonal90', 'di90': 'diagonal90', '5di90': 'diagonal90',
        }
        if raw in angle_map:
            return angle_map[raw]
    upper_name = image_path.stem.upper()
    for token, label in ANGLE_TOKEN_MAP.items():
        if token in upper_name:
            return label
    return None


def view_label_from_detail(angle_detail: str | None) -> str | None:
    if angle_detail == 'top':
        return 'top'
    if angle_detail in {'front45', 'diagonal45'}:
        return 'middle'
    if angle_detail in {'front90', 'diagonal90'}:
        return 'side'
    return None


def group_id_from_json_or_name(data: dict[str, Any] | None, image_path: Path, grade_label: str | None) -> str:
    if data and data.get('group_no') not in [None, '']:
        return str(data.get('group_no'))
    match = FILENAME_GROUP_PATTERN.search(image_path.name)
    if match:
        return f'{image_path.parent.name}_{grade_label}_{match.group(1)}'
    return image_path.stem


def bbox_from_json(data: dict[str, Any] | None) -> dict[str, int] | None:
    if not data or not isinstance(data.get('bndbox'), dict):
        return None
    box = data['bndbox']
    try:
        return {key: int(float(box[key])) for key in ['xmin', 'ymin', 'xmax', 'ymax']}
    except Exception:
        return None


def variety_from_json_or_path(data: dict[str, Any] | None, image_path: Path) -> str:
    if data and data.get('cate2'):
        return str(data.get('cate2'))
    text = normalized_text(str(image_path))
    if 'fuji' in text or '부사' in str(image_path):
        return '부사'
    if 'yanggwang' in text or '양광' in str(image_path):
        return '양광'
    return 'unknown'


def make_manifest(image_paths: list[Path], json_idx: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows, skipped = [], Counter()
    for image_path in image_paths:
        data = match_json_for_image(image_path, json_idx)
        grade_label = grade_from_json_or_path(data, image_path)
        angle_detail_label = angle_detail_from_json_or_name(data, image_path)
        angle_label = view_label_from_detail(angle_detail_label)
        if grade_label not in GRADE_LABELS:
            skipped['missing_grade'] += 1
            continue
        if angle_label not in ANGLE_LABELS:
            skipped['missing_angle'] += 1
            continue
        rows.append({
            'image_path': str(image_path),
            'json_path': data.get('_json_path') if data else None,
            'json_matched': data is not None,
            'source_split': official_split_from_path(image_path),
            'grade_label': grade_label,
            'angle_label': angle_label,
            'angle_detail_label': angle_detail_label,
            'variety': variety_from_json_or_path(data, image_path),
            'group_id': group_id_from_json_or_name(data, image_path, grade_label),
            'cate3': data.get('cate3') if data else None,
            'width_meta': data.get('width') if data else None,
            'height_meta': data.get('height') if data else None,
            'weight_meta': data.get('weight') if data else None,
            'bbox': bbox_from_json(data),
        })
    print('skipped:', dict(skipped))
    return pd.DataFrame(rows)


manifest = make_manifest(image_files, json_index)
print('manifest rows:', len(manifest))
display(manifest.head())
print('JSON matched ratio:', round(float(manifest['json_matched'].mean()), 4) if len(manifest) else 0)
print('등급 분포')
print(manifest['grade_label'].value_counts().sort_index())
print('각도 분포')
print(manifest['angle_label'].value_counts().sort_index())
print('품종 분포')
print(manifest['variety'].value_counts())


## 6. `group_no` 기준 2단계 데이터 분리

원본 데이터의 `Training` 폴더와 `Validation` 폴더는 먼저 하나로 합쳐서 사용합니다. 이후 `group_id` 기준으로 다시 분리합니다.

분리 순서:

```text
전체 group_id
-> train_valid 70% / test 30%
-> train_valid 내부에서 train 80% / valid 20%
```

최종 비율은 대략 아래와 같습니다.

```text
train 56% / valid 14% / test 30%
```

분리 단위는 이미지 1장이 아니라 `group_id`입니다. `group_id`는 JSON의 `group_no`를 우선 사용하고, 없으면 파일명에서 추출한 사과 번호를 사용합니다.

```text
같은 사과의 top/middle/side 사진 -> 하나의 split에만 배치
```

이 방식은 같은 사과를 여러 각도에서 찍은 이미지가 train과 test에 동시에 들어가는 문제를 줄입니다.


In [ ]:
def group_summary_table(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby('group_id')
        .agg(
            grade_label=('grade_label', 'first'),
            angle_label=('angle_label', lambda values: '|'.join(sorted(set(values)))),
            variety=('variety', 'first'),
            source_split=('source_split', lambda values: '|'.join(sorted(set(values)))),
        )
        .reset_index()
    )


def split_group_ids_once(
    group_table: pd.DataFrame,
    test_ratio: float,
    seed: int,
    stratify_columns: list[str],
) -> tuple[set[str], set[str]]:
    rng = random.Random(seed)
    left_ids: set[str] = set()
    right_ids: set[str] = set()

    groupby_columns = [column for column in stratify_columns if column in group_table.columns]
    grouped_iter = group_table.groupby(groupby_columns, dropna=False) if groupby_columns else [(None, group_table)]

    for _, grouped in grouped_iter:
        group_ids = list(grouped['group_id'])
        rng.shuffle(group_ids)
        n = len(group_ids)
        if n <= 1:
            left_ids.update(group_ids)
            continue
        n_right = int(round(n * test_ratio))
        n_right = max(1, min(n - 1, n_right))
        right_ids.update(group_ids[:n_right])
        left_ids.update(group_ids[n_right:])

    return left_ids, right_ids


def assign_splits(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['split'] = None
    group_table = group_summary_table(df)

    train_valid_group_ids, test_group_ids = split_group_ids_once(
        group_table,
        test_ratio=TEST_RATIO,
        seed=RANDOM_SEED,
        stratify_columns=['grade_label', 'variety'],
    )

    train_valid_group_table = group_table[group_table['group_id'].isin(train_valid_group_ids)].copy()
    train_group_ids, valid_group_ids = split_group_ids_once(
        train_valid_group_table,
        test_ratio=VALID_RATIO_IN_TRAIN_VALID,
        seed=RANDOM_SEED + 1,
        stratify_columns=['grade_label', 'variety'],
    )

    df.loc[df['group_id'].isin(train_group_ids), 'split'] = 'train'
    df.loc[df['group_id'].isin(valid_group_ids), 'split'] = 'valid'
    df.loc[df['group_id'].isin(test_group_ids), 'split'] = 'test'

    print('분리 방식: 공식 Training/Validation 폴더를 합친 뒤 group_id 기준 2단계 분리')
    print('1단계: train_valid', round(1 - TEST_RATIO, 3), '/ test', TEST_RATIO)
    print('2단계: train', round(1 - VALID_RATIO_IN_TRAIN_VALID, 3), '/ valid', VALID_RATIO_IN_TRAIN_VALID, '(train_valid 내부)')
    print('group 수:', {
        'train': len(train_group_ids),
        'valid': len(valid_group_ids),
        'test': len(test_group_ids),
    })

    return df[df['split'].notna()].reset_index(drop=True)


manifest = assign_splits(manifest)
print('split x grade')
print(pd.crosstab(manifest['split'], manifest['grade_label']))
print('split x angle')
print(pd.crosstab(manifest['split'], manifest['angle_label']))
print('split x variety')
print(pd.crosstab(manifest['split'], manifest['variety']))
print('split x source_split')
print(pd.crosstab(manifest['split'], manifest['source_split']))

leak_groups = manifest.groupby('group_id')['split'].nunique()
leak_groups = leak_groups[leak_groups > 1]
print('group_id split 누수 개수:', len(leak_groups))
if len(leak_groups):
    print('주의: 같은 group_id가 여러 split에 있습니다.')
else:
    print('같은 group_id는 하나의 split에만 배치되었습니다.')


## 7. 이미지 품질 지표와 보조 특징

밝기, 과노출, 흐림, 색 선명도, 둥근 정도, 멍 의심 확률은 JSON 없이 이미지 픽셀에서 계산합니다.

In [ ]:
def load_rgb(image_path: str | Path, image_size: int | None = None, bbox: dict[str, int] | None = None) -> Image.Image:
    image = Image.open(image_path).convert('RGB')
    if bbox and USE_BBOX_CROP:
        w, h = image.size
        pad_x = int((bbox['xmax'] - bbox['xmin']) * BBOX_PADDING_RATIO)
        pad_y = int((bbox['ymax'] - bbox['ymin']) * BBOX_PADDING_RATIO)
        left, top = max(0, bbox['xmin'] - pad_x), max(0, bbox['ymin'] - pad_y)
        right, bottom = min(w, bbox['xmax'] + pad_x), min(h, bbox['ymax'] + pad_y)
        if right > left and bottom > top:
            image = image.crop((left, top, right, bottom))
    if image_size:
        image = image.resize((image_size, image_size))
    return image


def image_quality_metrics(image_path: str | Path, bbox: dict[str, int] | None = None) -> dict[str, float]:
    image = load_rgb(image_path, image_size=DEFAULT_IMAGE_SIZE, bbox=bbox)
    hsv = np.asarray(image.convert('HSV'), dtype=np.float32)
    value = hsv[..., 2] / 255.0
    saturation = hsv[..., 1] / 255.0
    edges = np.asarray(image.convert('L').filter(ImageFilter.FIND_EDGES), dtype=np.float32) / 255.0
    return {
        'brightness': round(float(value.mean()), 4),
        'underexposed_ratio': round(float((value < 0.18).mean()), 4),
        'overexposed_ratio': round(float((value > 0.95).mean()), 4),
        'saturation_mean': round(float(saturation.mean()), 4),
        'blur_score': round(float(edges.std()), 4),
    }


def calculate_color_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE, bbox: dict[str, int] | None = None) -> float:
    hsv = np.asarray(load_rgb(image_path, image_size=image_size, bbox=bbox).convert('HSV'), dtype=np.float32)
    saturation = hsv[..., 1] / 255.0
    value = hsv[..., 2] / 255.0
    score = (saturation.mean() * 0.65 + value.mean() * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_roundness_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE, bbox: dict[str, int] | None = None) -> float:
    image = load_rgb(image_path, image_size=image_size, bbox=bbox)
    arr = np.asarray(image, dtype=np.float32)
    border = np.concatenate([arr[:8].reshape(-1, 3), arr[-8:].reshape(-1, 3), arr[:, :8].reshape(-1, 3), arr[:, -8:].reshape(-1, 3)], axis=0)
    bg = np.median(border, axis=0)
    distance = np.linalg.norm(arr - bg, axis=2)
    mask = distance > max(18.0, float(distance.mean()))
    ys, xs = np.where(mask)
    if len(xs) < 50:
        return 50.0
    width, height = max(1, xs.max() - xs.min() + 1), max(1, ys.max() - ys.min() + 1)
    aspect_score = min(width, height) / max(width, height)
    fill_ratio = mask.sum() / float(width * height)
    score = (aspect_score * 0.65 + min(fill_ratio / 0.78, 1.0) * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_bruise_probability(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE, bbox: dict[str, int] | None = None) -> float:
    hsv = np.asarray(load_rgb(image_path, image_size=image_size, bbox=bbox).convert('HSV'), dtype=np.float32)
    value = hsv[..., 2] / 255.0
    saturation = hsv[..., 1] / 255.0
    dark_ratio = np.logical_and(value < 0.33, saturation > 0.18).mean()
    return round(float(np.clip(min(1.0, dark_ratio / 0.18), 0.0, 1.0)), 4)


if len(manifest):
    display(pd.DataFrame([image_quality_metrics(row.image_path, row.bbox) for row in manifest.head(20).itertuples(index=False)]).describe())

## 8. 데이터셋과 데이터 로더

view 모델은 `top/middle/side` 3분류로 학습합니다.

이 노트에서는 train 데이터에만 balanced dataset을 적용합니다. valid/test는 실제 평가 분포를 보기 위해 원본 분포를 유지합니다.

균형 기준:

```text
view 모델 train:
angle_label x grade_label x variety 조합 개수 동일화

grade 모델 train:
grade_label x variety 조합 개수 동일화

valid/test:
원본 분포 유지
```

즉, view 모델은 `top/middle/side`뿐 아니라 A/B/C 등급과 양광/부사 품종도 함께 균형을 맞춥니다. top/middle/side 각각의 등급 모델도 A/B/C와 품종이 같은 비율로 들어가도록 train dataset을 구성합니다.

샘플 선택은 이미지 단위 랜덤이 아니라 `group_id` 우선 방식으로 수행해 가능한 한 다양한 사과가 포함되도록 합니다.

등급 모델은 시점별로 분리합니다.

```text
top_grade 모델: top 사진만 사용해 A/B/C 학습
middle_grade 모델: middle 사진만 사용해 A/B/C 학습
side_grade 모델: side 사진만 사용해 A/B/C 학습
```


In [ ]:
def task_label_col(task: str) -> str:
    if task == 'angle':
        return 'angle_label'
    if task in {'top_grade', 'middle_grade', 'side_grade'}:
        return 'grade_label'
    raise ValueError(f'Unknown task: {task}')


def task_angle_filter(task: str) -> str | None:
    if task == 'top_grade':
        return 'top'
    if task == 'middle_grade':
        return 'middle'
    if task == 'side_grade':
        return 'side'
    return None


def train_balance_target_count(counts: pd.Series) -> int:
    if isinstance(TRAIN_BALANCE_TARGET, int):
        return int(TRAIN_BALANCE_TARGET)
    if TRAIN_BALANCE_TARGET == 'min':
        return int(counts.min())
    if TRAIN_BALANCE_TARGET == 'max':
        return int(counts.max())
    raise ValueError(f'지원하지 않는 TRAIN_BALANCE_TARGET: {TRAIN_BALANCE_TARGET}')


def sample_group_aware(part: pd.DataFrame, target_count: int, seed: int) -> pd.DataFrame:
    if part.empty:
        return part
    if not TRAIN_BALANCE_GROUP_AWARE or 'group_id' not in part.columns:
        replace = len(part) < target_count
        return part.sample(n=target_count, replace=replace, random_state=seed)

    rng = random.Random(seed)
    selected_indices: list[int] = []
    group_to_indices: dict[str, list[int]] = {}
    for group_id, group_df in part.groupby('group_id', sort=False):
        indices = list(group_df.index)
        rng.shuffle(indices)
        group_to_indices[str(group_id)] = indices

    group_ids = list(group_to_indices.keys())
    rng.shuffle(group_ids)

    # 1차: group_id를 순회하면서 group당 제한 개수만큼 먼저 선택합니다.
    for round_index in range(max(1, TRAIN_BALANCE_MAX_IMAGES_PER_GROUP)):
        for group_id in group_ids:
            indices = group_to_indices[group_id]
            if round_index < len(indices):
                selected_indices.append(indices[round_index])
                if len(selected_indices) >= target_count:
                    break
        if len(selected_indices) >= target_count:
            break

    # 2차: 아직 target_count가 부족하면 남은 이미지를 group 순회 방식으로 추가합니다.
    round_index = max(1, TRAIN_BALANCE_MAX_IMAGES_PER_GROUP)
    while len(selected_indices) < target_count:
        added = False
        for group_id in group_ids:
            indices = group_to_indices[group_id]
            if round_index < len(indices):
                selected_indices.append(indices[round_index])
                added = True
                if len(selected_indices) >= target_count:
                    break
        if not added:
            break
        round_index += 1

    # 3차: 원본 이미지 수가 target_count보다 적을 때만 중복 허용으로 채웁니다.
    if len(selected_indices) < target_count:
        pool = list(part.index)
        while len(selected_indices) < target_count:
            selected_indices.append(rng.choice(pool))

    return part.loc[selected_indices].copy()


def make_balance_key_series(df: pd.DataFrame, columns: list[str]) -> pd.Series:
    return df[columns].astype(str).agg('|'.join, axis=1)


def make_train_balanced_df(df: pd.DataFrame, task: str) -> pd.DataFrame:
    if df.empty or not USE_TRAIN_BALANCED_DATASET:
        return df
    columns = BALANCE_GROUP_COLUMNS.get(task)
    if not columns:
        return df
    missing_columns = [column for column in columns if column not in df.columns]
    if missing_columns:
        print(f'[{task}] balanced dataset을 만들 수 없어 원본 train 데이터를 사용합니다. 누락 컬럼:', missing_columns)
        return df

    df = df.copy()
    df['_balance_key'] = make_balance_key_series(df, columns)
    counts = df['_balance_key'].value_counts().sort_index()
    if counts.empty:
        return df.drop(columns=['_balance_key'])

    target_count = max(1, train_balance_target_count(counts))
    balanced_parts = []
    rng_seed = TRAIN_BALANCE_RANDOM_SEED
    for key in counts.index:
        part = df[df['_balance_key'] == key]
        sampled = sample_group_aware(part, target_count, rng_seed)
        balanced_parts.append(sampled)

    balanced_df = pd.concat(balanced_parts, axis=0).sample(frac=1.0, random_state=rng_seed).reset_index(drop=True)
    print(f'[{task}] train balance 기준 컬럼:', columns)
    print(f'[{task}] train 원본 조합 분포:', counts.to_dict())
    print(f'[{task}] train balanced 조합 분포:', balanced_df['_balance_key'].value_counts().sort_index().to_dict())
    print(f'[{task}] train balanced 등급 분포:', balanced_df['grade_label'].value_counts().sort_index().to_dict())
    print(f'[{task}] train balanced 품종 분포:', balanced_df['variety'].value_counts().sort_index().to_dict())
    if 'angle_label' in balanced_df.columns:
        print(f'[{task}] train balanced view 분포:', balanced_df['angle_label'].value_counts().sort_index().to_dict())
    if 'group_id' in balanced_df.columns:
        print(f'[{task}] train balanced group 수:', balanced_df.groupby('_balance_key')['group_id'].nunique().to_dict())
    return balanced_df.drop(columns=['_balance_key'])


def build_transforms(image_size: int = DEFAULT_IMAGE_SIZE, train: bool = True, task: str = 'top_grade'):
    if train and task == 'angle':
        return transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.78, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(8),
            transforms.ColorJitter(brightness=0.25, contrast=0.22, saturation=0.18),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.82, 1.0), ratio=(0.92, 1.08)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(12),
            transforms.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])


class AppleManifestDataset(Dataset):
    def __init__(self, manifest_df: pd.DataFrame, split: str, task: str, class_names: list[str], transform=None):
        self.task = task
        self.class_names = list(class_names)
        self.class_to_idx = {label: idx for idx, label in enumerate(self.class_names)}
        self.transform = transform
        self.label_col = task_label_col(task)
        df = manifest_df[manifest_df['split'] == split].copy()
        angle_filter = task_angle_filter(task)
        if angle_filter is not None:
            df = df[df['angle_label'] == angle_filter]
        df = df[df[self.label_col].isin(self.class_to_idx)]
        if split == 'train':
            df = make_train_balanced_df(df, task)
        self.records = df.to_dict('records')
        self.targets = [self.class_to_idx[record[self.label_col]] for record in self.records]

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        image = load_rgb(record['image_path'], bbox=record.get('bbox'))
        if self.transform is not None:
            image = self.transform(image)
        target = self.class_to_idx[record[self.label_col]]
        return image, target


def balance_key(record: dict[str, Any], task: str) -> str:
    columns = BALANCE_GROUP_COLUMNS.get(task, [task_label_col(task)])
    return '|'.join(str(record.get(column, 'unknown')) for column in columns)


def build_balanced_sampler(dataset: AppleManifestDataset, task: str) -> WeightedRandomSampler:
    keys = [balance_key(record, task) for record in dataset.records]
    group_counts = Counter(keys)
    if not group_counts:
        raise ValueError(f'{task} sampler를 만들 데이터가 없습니다.')
    max_count = max(group_counts.values())
    target_count = max_count if BALANCED_SAMPLES_PER_GROUP == 'max' else int(BALANCED_SAMPLES_PER_GROUP)
    weights = []
    effective_counts = Counter()
    for key in keys:
        count = group_counts[key]
        capped_target = min(target_count, int(count * MAX_OVERSAMPLE_MULTIPLIER))
        capped_target = max(capped_target, count)
        weights.append(capped_target / count)
        effective_counts[key] = capped_target
    num_samples = int(sum(effective_counts.values()))
    print(f'[{task}] balanced sampler 기준 컬럼:', BALANCE_GROUP_COLUMNS.get(task))
    print(f'[{task}] 원본 조합 수:', len(group_counts), '샘플 수:', len(keys), 'epoch 샘플 수:', num_samples)
    print(f'[{task}] 적은 조합 예시:', group_counts.most_common()[-8:])
    return WeightedRandomSampler(weights=weights, num_samples=num_samples, replacement=True)


def build_loader(task: str, split: str, class_names: list[str], image_size: int, batch_size: int, num_workers: int) -> DataLoader:
    dataset = AppleManifestDataset(manifest, split=split, task=task, class_names=class_names, transform=build_transforms(image_size=image_size, train=(split == 'train'), task=task))
    if len(dataset) == 0:
        raise ValueError(f'{task}/{split} 데이터가 없습니다.')
    sampler, shuffle = None, split == 'train'
    if split == 'train' and USE_WEIGHTED_SAMPLER:
        sampler = build_balanced_sampler(dataset, task)
        shuffle = False
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, sampler=sampler, num_workers=num_workers, pin_memory=torch.cuda.is_available())


for task, classes in [('angle', ANGLE_LABELS), ('top_grade', GRADE_LABELS), ('middle_grade', GRADE_LABELS), ('side_grade', GRADE_LABELS)]:
    print(f'[{task}]')
    for split in ['train', 'valid', 'test']:
        ds = AppleManifestDataset(manifest, split=split, task=task, class_names=classes, transform=None)
        print(split, len(ds), Counter(ds.targets))

print('\nangle detail 분포')
print(pd.crosstab(manifest['split'], manifest['angle_detail_label']))
print('\ntop/middle/side 분포')
print(pd.crosstab(manifest['split'], manifest['angle_label']))


## 9. ResNet18 모델과 공통 학습 함수

각도 모델과 등급 모델은 같은 ResNet18 구조를 사용하고, 마지막 출력층만 클래스 개수에 맞게 바꿉니다.

In [ ]:
def build_resnet18(num_classes: int, pretrained: bool = False) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def evaluate_model(model: nn.Module, data_loader: DataLoader, criterion, device: torch.device) -> dict[str, Any]:
    model.eval()
    total_loss, y_true, y_pred = 0.0, [], []
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += float(criterion(outputs, targets).item())
            predicted = outputs.argmax(dim=1)
            y_true.extend(targets.detach().cpu().tolist())
            y_pred.extend(predicted.detach().cpu().tolist())
    return {
        'loss': total_loss / max(1, len(data_loader)),
        'accuracy': accuracy_score(y_true, y_pred) if y_true else 0.0,
        'f1': f1_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'y_true': y_true,
        'y_pred': y_pred,
    }


class FocalLoss(nn.Module):
    def __init__(self, weight: torch.Tensor | None = None, gamma: float = FOCAL_GAMMA):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


def class_weights_from_dataset(dataset: AppleManifestDataset, num_classes: int, device: torch.device) -> torch.Tensor | None:
    if not USE_CLASS_WEIGHTED_LOSS:
        return None
    counts = Counter(dataset.targets)
    total = sum(counts.values())
    weights = []
    for index in range(num_classes):
        count = max(1, counts.get(index, 0))
        weight = total / (num_classes * count)
        weights.append(min(float(weight), MAX_CLASS_LOSS_WEIGHT))
    tensor = torch.tensor(weights, dtype=torch.float32, device=device)
    tensor = tensor / tensor.mean()
    print('class loss weights:', [round(float(v), 4) for v in tensor.detach().cpu().tolist()])
    return tensor


def build_criterion(train_dataset: AppleManifestDataset, num_classes: int, device: torch.device):
    weights = class_weights_from_dataset(train_dataset, num_classes, device)
    if USE_FOCAL_LOSS:
        print('loss: FocalLoss')
        return FocalLoss(weight=weights, gamma=FOCAL_GAMMA)
    print('loss: CrossEntropyLoss')
    return nn.CrossEntropyLoss(weight=weights)


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion, optimizer, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
    return total_loss / max(1, len(loader))


def train_task_model(task: str, class_names: list[str], checkpoint_name: str, pretrained: bool = USE_PRETRAINED):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cfg = TRAIN_CONFIG
    train_loader = build_loader(task, 'train', class_names, cfg['image_size'], cfg['batch_size'], cfg['num_workers'])
    valid_loader = build_loader(task, 'valid', class_names, cfg['image_size'], cfg['batch_size'], cfg['num_workers'])
    test_loader = build_loader(task, 'test', class_names, cfg['image_size'], cfg['batch_size'], cfg['num_workers'])

    model = build_resnet18(num_classes=len(class_names), pretrained=pretrained).to(device)
    criterion = build_criterion(train_loader.dataset, len(class_names), device)
    optimizer = optim.Adam(model.parameters(), lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])

    best_f1, best_state, best_checkpoint, wait = -1.0, None, None, 0
    checkpoint_path = MODEL_ROOT / checkpoint_name
    print(f'[{task}] device={device} pretrained={pretrained}')

    for epoch in range(1, cfg['epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        valid_metrics = evaluate_model(model, valid_loader, criterion, device)
        print(f"epoch {epoch:02d} train_loss={train_loss:.4f} valid_loss={valid_metrics['loss']:.4f} valid_acc={valid_metrics['accuracy']:.4f} valid_f1={valid_metrics['f1']:.4f}")
        if valid_metrics['f1'] > best_f1:
            best_f1 = valid_metrics['f1']
            best_state = copy.deepcopy(model.state_dict())
            best_checkpoint = {
                'task': task,
                'fruit_type': TARGET_FRUIT,
                'model_name': DEFAULT_MODEL_NAME,
                'model_state_dict': best_state,
                'labels': class_names,
                'image_size': cfg['image_size'],
                'model_version': f'{TARGET_FRUIT}-{task}-resnet18-v0',
                'train_config': cfg,
                'balance_group_columns': BALANCE_GROUP_COLUMNS.get(task),
                'max_oversample_multiplier': MAX_OVERSAMPLE_MULTIPLIER,
                'use_class_weighted_loss': USE_CLASS_WEIGHTED_LOSS,
                'use_focal_loss': USE_FOCAL_LOSS,
                'focal_gamma': FOCAL_GAMMA,
                'valid_metrics': {k: v for k, v in valid_metrics.items() if k not in {'y_true', 'y_pred'}},
                'use_bbox_crop': USE_BBOX_CROP,
            }
            torch.save(best_checkpoint, checkpoint_path)
            wait = 0
        else:
            wait += 1
        if wait >= cfg['patience']:
            print('early stopping')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader, criterion, device)
    best_checkpoint['test_metrics'] = {k: v for k, v in test_metrics.items() if k not in {'y_true', 'y_pred'}}
    torch.save(best_checkpoint, checkpoint_path)
    print(f"[{task}] test_acc={test_metrics['accuracy']:.4f} test_f1={test_metrics['f1']:.4f}")
    print('saved:', checkpoint_path)
    return checkpoint_path, test_metrics


print('angle output:', build_resnet18(len(ANGLE_LABELS), pretrained=False).fc)
print('grade output:', build_resnet18(len(GRADE_LABELS), pretrained=False).fc)

## 10. top/middle/side 확인 모델 학습

view 모델은 세부 각도 5개를 그대로 맞히는 것이 아니라, 서비스 추론에 필요한 `top/middle/side` 3개 시점으로 이미지를 분류합니다.

```text
top -> top
front45, diagonal45 -> middle
front90, diagonal90 -> side
```

이 노트에서는 view 모델 train 데이터에만 balanced dataset을 적용해 top 쏠림을 줄입니다.


In [ ]:
ANGLE_CHECKPOINT_PATH = None
ANGLE_TEST_METRICS = None

if RUN_ANGLE_TRAINING:
    try:
        ANGLE_CHECKPOINT_PATH, ANGLE_TEST_METRICS = train_task_model('angle', ANGLE_LABELS, 'apple_view_top_middle_side_balanced_resnet18_best.pt', pretrained=USE_PRETRAINED)
    except Exception as exc:
        if USE_PRETRAINED:
            print('사전학습 가중치 사용 실패. 사전학습 없이 top/middle/side 모델을 다시 학습합니다:', repr(exc))
            ANGLE_CHECKPOINT_PATH, ANGLE_TEST_METRICS = train_task_model('angle', ANGLE_LABELS, 'apple_view_top_middle_side_balanced_resnet18_best.pt', pretrained=False)
        else:
            raise
else:
    print('RUN_ANGLE_TRAINING=False라 top/middle/side 모델 학습을 건너뜁니다.')


## 11. top/middle/side 등급 모델 학습

top, middle, side 사진은 보이는 정보가 다르므로 등급 모델을 분리합니다.

```text
apple_top_grade_resnet18_best.pt
apple_middle_grade_resnet18_best.pt
apple_side_grade_resnet18_best.pt
```


In [ ]:
TOP_GRADE_CHECKPOINT_PATH = None
TOP_GRADE_TEST_METRICS = None
MIDDLE_GRADE_CHECKPOINT_PATH = None
MIDDLE_GRADE_TEST_METRICS = None
SIDE_GRADE_CHECKPOINT_PATH = None
SIDE_GRADE_TEST_METRICS = None

if RUN_TOP_GRADE_TRAINING:
    try:
        TOP_GRADE_CHECKPOINT_PATH, TOP_GRADE_TEST_METRICS = train_task_model('top_grade', GRADE_LABELS, 'apple_top_grade_resnet18_best.pt', pretrained=USE_PRETRAINED)
    except Exception as exc:
        if USE_PRETRAINED:
            print('사전학습 가중치 사용 실패. 사전학습 없이 top 등급 모델을 다시 학습합니다:', repr(exc))
            TOP_GRADE_CHECKPOINT_PATH, TOP_GRADE_TEST_METRICS = train_task_model('top_grade', GRADE_LABELS, 'apple_top_grade_resnet18_best.pt', pretrained=False)
        else:
            raise
else:
    print('RUN_TOP_GRADE_TRAINING=False라 top 등급 모델 학습을 건너뜁니다.')

if RUN_MIDDLE_GRADE_TRAINING:
    try:
        MIDDLE_GRADE_CHECKPOINT_PATH, MIDDLE_GRADE_TEST_METRICS = train_task_model('middle_grade', GRADE_LABELS, 'apple_middle_grade_resnet18_best.pt', pretrained=USE_PRETRAINED)
    except Exception as exc:
        if USE_PRETRAINED:
            print('사전학습 가중치 사용 실패. 사전학습 없이 middle 등급 모델을 다시 학습합니다:', repr(exc))
            MIDDLE_GRADE_CHECKPOINT_PATH, MIDDLE_GRADE_TEST_METRICS = train_task_model('middle_grade', GRADE_LABELS, 'apple_middle_grade_resnet18_best.pt', pretrained=False)
        else:
            raise
else:
    print('RUN_MIDDLE_GRADE_TRAINING=False라 middle 등급 모델 학습을 건너뜁니다.')

if RUN_SIDE_GRADE_TRAINING:
    try:
        SIDE_GRADE_CHECKPOINT_PATH, SIDE_GRADE_TEST_METRICS = train_task_model('side_grade', GRADE_LABELS, 'apple_side_grade_resnet18_best.pt', pretrained=USE_PRETRAINED)
    except Exception as exc:
        if USE_PRETRAINED:
            print('사전학습 가중치 사용 실패. 사전학습 없이 side 등급 모델을 다시 학습합니다:', repr(exc))
            SIDE_GRADE_CHECKPOINT_PATH, SIDE_GRADE_TEST_METRICS = train_task_model('side_grade', GRADE_LABELS, 'apple_side_grade_resnet18_best.pt', pretrained=False)
        else:
            raise
else:
    print('RUN_SIDE_GRADE_TRAINING=False라 side 등급 모델 학습을 건너뜁니다.')


## 12. 평가 리포트

top/middle/side 확인 모델과 각 시점별 등급 모델을 각각 평가합니다. 최종 서비스에서는 이미지 1장을 받아 view 모델 결과에 따라 적절한 등급 모델을 자동으로 선택합니다.


In [ ]:
def load_checkpoint_model(checkpoint_path: Path, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    labels = checkpoint['labels']
    model = build_resnet18(num_classes=len(labels), pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, checkpoint


def predict_records(checkpoint_path: Path, task: str, class_names: list[str], split: str = 'test') -> pd.DataFrame:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, _ = load_checkpoint_model(checkpoint_path, device)
    dataset = AppleManifestDataset(manifest, split=split, task=task, class_names=class_names, transform=build_transforms(train=False, task=task))
    loader = DataLoader(dataset, batch_size=TRAIN_CONFIG['batch_size'], shuffle=False, num_workers=TRAIN_CONFIG['num_workers'])
    rows, offset = [], 0
    with torch.no_grad():
        for inputs, targets in loader:
            probs = torch.softmax(model(inputs.to(device)), dim=1).detach().cpu().numpy()
            preds = probs.argmax(axis=1)
            for i, pred_idx in enumerate(preds):
                record = dataset.records[offset + i]
                true_idx = int(targets[i])
                rows.append({**record, 'true_label': class_names[true_idx], 'pred_label': class_names[int(pred_idx)], 'confidence': float(probs[i, pred_idx])})
            offset += len(targets)
    return pd.DataFrame(rows)


def print_report(pred_df: pd.DataFrame, labels: list[str], title: str):
    print('\n=', title)
    if pred_df.empty:
        print('평가 데이터가 없습니다.')
        return
    print('accuracy:', round(accuracy_score(pred_df['true_label'], pred_df['pred_label']), 4))
    print('macro_f1:', round(f1_score(pred_df['true_label'], pred_df['pred_label'], average='macro', zero_division=0), 4))
    print(pd.DataFrame(confusion_matrix(pred_df['true_label'], pred_df['pred_label'], labels=labels), index=labels, columns=labels))
    print(classification_report(pred_df['true_label'], pred_df['pred_label'], labels=labels, zero_division=0))


def grouped_accuracy(pred_df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    if pred_df.empty or group_col not in pred_df:
        return pd.DataFrame()
    return pd.DataFrame([
        {group_col: value, 'count': len(group), 'accuracy': accuracy_score(group['true_label'], group['pred_label']), 'macro_f1': f1_score(group['true_label'], group['pred_label'], average='macro', zero_division=0)}
        for value, group in pred_df.groupby(group_col)
    ]).sort_values('count', ascending=False)


if ANGLE_CHECKPOINT_PATH is None and (MODEL_ROOT / 'apple_view_top_middle_side_balanced_resnet18_best.pt').exists():
    ANGLE_CHECKPOINT_PATH = MODEL_ROOT / 'apple_view_top_middle_side_balanced_resnet18_best.pt'
if TOP_GRADE_CHECKPOINT_PATH is None and (MODEL_ROOT / 'apple_top_grade_resnet18_best.pt').exists():
    TOP_GRADE_CHECKPOINT_PATH = MODEL_ROOT / 'apple_top_grade_resnet18_best.pt'
if MIDDLE_GRADE_CHECKPOINT_PATH is None and (MODEL_ROOT / 'apple_middle_grade_resnet18_best.pt').exists():
    MIDDLE_GRADE_CHECKPOINT_PATH = MODEL_ROOT / 'apple_middle_grade_resnet18_best.pt'
if SIDE_GRADE_CHECKPOINT_PATH is None and (MODEL_ROOT / 'apple_side_grade_resnet18_best.pt').exists():
    SIDE_GRADE_CHECKPOINT_PATH = MODEL_ROOT / 'apple_side_grade_resnet18_best.pt'

if ANGLE_CHECKPOINT_PATH:
    angle_pred_df = predict_records(Path(ANGLE_CHECKPOINT_PATH), 'angle', ANGLE_LABELS, split='test')
    print_report(angle_pred_df, ANGLE_LABELS, 'top/middle/side 확인 모델 test 평가')

if TOP_GRADE_CHECKPOINT_PATH:
    top_grade_pred_df = predict_records(Path(TOP_GRADE_CHECKPOINT_PATH), 'top_grade', GRADE_LABELS, split='test')
    print_report(top_grade_pred_df, GRADE_LABELS, 'top 등급 모델 test 평가')
    print('top 등급 모델 품종별 성능')
    display(grouped_accuracy(top_grade_pred_df, 'variety'))

if MIDDLE_GRADE_CHECKPOINT_PATH:
    middle_grade_pred_df = predict_records(Path(MIDDLE_GRADE_CHECKPOINT_PATH), 'middle_grade', GRADE_LABELS, split='test')
    print_report(middle_grade_pred_df, GRADE_LABELS, 'middle 등급 모델 test 평가')
    print('middle 등급 모델 세부 각도별 성능')
    display(grouped_accuracy(middle_grade_pred_df, 'angle_detail_label'))
    print('middle 등급 모델 품종별 성능')
    display(grouped_accuracy(middle_grade_pred_df, 'variety'))

if SIDE_GRADE_CHECKPOINT_PATH:
    side_grade_pred_df = predict_records(Path(SIDE_GRADE_CHECKPOINT_PATH), 'side_grade', GRADE_LABELS, split='test')
    print_report(side_grade_pred_df, GRADE_LABELS, 'side 등급 모델 test 평가')
    print('side 등급 모델 세부 각도별 성능')
    display(grouped_accuracy(side_grade_pred_df, 'angle_detail_label'))
    print('side 등급 모델 품종별 성능')
    display(grouped_accuracy(side_grade_pred_df, 'variety'))


## 13. 단일 이미지 자동 분기 추론

실제 서비스에서는 사용자가 사과 사진 1장만 업로드합니다. 서버는 먼저 view 모델로 이미지가 `top`, `middle`, `side` 중 어디에 가까운지 판단하고, 판단된 시점에 맞는 등급 모델을 자동으로 선택합니다.

이 방식은 앱과 API를 단일 이미지 업로드 구조로 유지하면서도, 애매한 중간/사선 시점을 side와 분리해 학습할 수 있습니다.


In [ ]:
@dataclass(frozen=True)
class AppleFreshnessSingleResult:
    fruit_type: str
    angle_label: str
    angle_confidence: float
    model_grade: str
    grade_confidence: float
    color_score: float
    roundness_score: float
    bruise_probability: float
    image_quality: dict[str, float]
    freshness_score: float
    model_decision: str
    model_version: str


def calculate_freshness_score(model_grade: str, color_score: float, roundness_score: float, bruise_probability: float) -> float:
    grade_score = GRADE_SCORE_MAP.get(model_grade, GRADE_SCORE_MAP['C'])
    bruise_free_score = max(0.0, min(100.0, (1.0 - bruise_probability) * 100.0))
    score = grade_score * 0.60 + color_score * 0.20 + roundness_score * 0.10 + bruise_free_score * 0.10
    return round(max(0.0, min(100.0, score)), 2)


def make_single_decision(result: AppleFreshnessSingleResult) -> str:
    if result.angle_confidence < 0.60:
        return 'REVIEW'
    if result.grade_confidence < 0.55:
        return 'REVIEW'
    if result.freshness_score < 60.0 or result.bruise_probability >= 0.5:
        return 'HOLD'
    if result.freshness_score >= 80.0 and result.model_grade == 'A':
        return 'PASS'
    return 'REVIEW'


def predict_with_checkpoint(image_path: str | Path, checkpoint_path: Path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, checkpoint = load_checkpoint_model(checkpoint_path, device)
    labels = checkpoint['labels']
    image_size = int(checkpoint.get('image_size', DEFAULT_IMAGE_SIZE))
    task = checkpoint.get('task', 'top_grade')
    tensor = build_transforms(image_size=image_size, train=False, task=task)(load_rgb(image_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    confidence, index = torch.max(probs, dim=0)
    return labels[int(index.item())], round(float(confidence.item()), 4), checkpoint


def grade_checkpoint_for_angle(angle_label: str) -> Path:
    if angle_label == 'top':
        return MODEL_ROOT / 'apple_top_grade_resnet18_best.pt'
    if angle_label == 'middle':
        return MODEL_ROOT / 'apple_middle_grade_resnet18_best.pt'
    return MODEL_ROOT / 'apple_side_grade_resnet18_best.pt'


def predict_apple_single(image_path: str | Path) -> AppleFreshnessSingleResult:
    image_path = Path(image_path)
    angle_checkpoint = MODEL_ROOT / 'apple_view_top_middle_side_balanced_resnet18_best.pt'

    angle_label, angle_confidence, _ = predict_with_checkpoint(image_path, angle_checkpoint)
    grade_checkpoint = grade_checkpoint_for_angle(angle_label)
    model_grade, grade_confidence, _ = predict_with_checkpoint(image_path, grade_checkpoint)

    color_score = calculate_color_score(image_path)
    roundness_score = calculate_roundness_score(image_path)
    bruise_probability = calculate_bruise_probability(image_path)
    freshness_score = calculate_freshness_score(
        model_grade=model_grade,
        color_score=color_score,
        roundness_score=roundness_score,
        bruise_probability=bruise_probability,
    )

    result = AppleFreshnessSingleResult(
        fruit_type=TARGET_FRUIT,
        angle_label=angle_label,
        angle_confidence=angle_confidence,
        model_grade=model_grade,
        grade_confidence=grade_confidence,
        color_score=color_score,
        roundness_score=roundness_score,
        bruise_probability=bruise_probability,
        image_quality=image_quality_metrics(image_path),
        freshness_score=freshness_score,
        model_decision='REVIEW',
        model_version='apple-single-image-top-middle-side-view-balanced-router-v0',
    )
    return AppleFreshnessSingleResult(**{**asdict(result), 'model_decision': make_single_decision(result)})


single_candidates = manifest[manifest['split'] == 'test']
required_checkpoints = [
    'apple_view_top_middle_side_balanced_resnet18_best.pt',
    'apple_top_grade_resnet18_best.pt',
    'apple_middle_grade_resnet18_best.pt',
    'apple_side_grade_resnet18_best.pt',
]
if len(single_candidates) and all((MODEL_ROOT / name).exists() for name in required_checkpoints):
    print(asdict(predict_apple_single(single_candidates.iloc[0]['image_path'])))
else:
    print('단일 이미지 추론 예시는 필요한 체크포인트 또는 샘플 이미지가 없어 건너뜁니다.')


## 14. FastAPI 응답 변환

FastAPI에서는 이미지 파일 1장을 업로드받아 저장한 뒤 `predict_apple_single(image_path)`에 전달하면 됩니다.

응답 구조는 Docs의 공통 응답 형식인 `data`, `message`, `error`를 따릅니다.


In [ ]:
def to_api_response(result: AppleFreshnessSingleResult, quality_inspection_id: int | None = None, image_url: str | None = None) -> dict[str, Any]:
    return {
        'data': {
            'quality_inspection_id': quality_inspection_id,
            'fruit_type': result.fruit_type,
            'image_url': image_url,
            'angle_label': result.angle_label,
            'angle_confidence': result.angle_confidence,
            'model_grade': result.model_grade,
            'grade_confidence': result.grade_confidence,
            'freshness_score': result.freshness_score,
            'color_score': result.color_score,
            'roundness_score': result.roundness_score,
            'bruise_probability': result.bruise_probability,
            'model_decision': result.model_decision,
            'model_version': result.model_version,
            'image_quality': result.image_quality,
        },
        'message': 'success',
        'error': None,
    }


## 최종 정리

이 노트북은 Kaggle Notebook에 사과 데이터셋만 연결하면 실행할 수 있는 단일 이미지 입력 기반 학습 파이프라인입니다.

이 버전의 가장 큰 차이는 view 모델 학습 방식입니다.

```text
top -> top
front45, diagonal45 -> middle
front90, diagonal90 -> side
```

데이터 분리는 원본 `Training`과 `Validation` 폴더를 먼저 합친 뒤, 이미지 행 단위 random split이 아니라 `group_no` 또는 사과 번호 기준으로 수행합니다.

```text
같은 사과의 여러 각도 사진 -> train/valid/test 중 하나에만 배치
```

그리고 모든 모델의 train 데이터는 A/B/C 등급과 양광/부사 품종 개수가 동일해지도록 balanced dataset으로 학습합니다. view 모델은 추가로 top/middle/side까지 함께 균형을 맞춥니다. 샘플 선택은 `group_id` 우선 방식으로 수행해 가능한 한 다양한 사과가 포함되도록 합니다.

```text
view 모델 train -> angle_label x grade_label x variety group-aware balanced
grade 모델 train -> grade_label x variety group-aware balanced
valid/test -> 원본 분포 유지
```

핵심 흐름:

```text
JSON cate3 -> A/B/C 등급 라벨 생성
JSON angle_direction 또는 파일명 -> top/middle/side 라벨 생성
JSON group_no 또는 파일명 번호 -> group_id 생성
Training/Validation 합치기 -> group_id 기준 7:3 test 분리 -> 남은 train_valid에서 8:2 train/valid 분리
balanced train dataset -> top/middle/side 확인 모델과 각 grade 모델 학습
top/middle/side 확인 모델 -> apple_view_top_middle_side_balanced_resnet18_best.pt
top 등급 모델 -> apple_top_grade_resnet18_best.pt
middle 등급 모델 -> apple_middle_grade_resnet18_best.pt
side 등급 모델 -> apple_side_grade_resnet18_best.pt
단일 이미지 추론 -> view 모델 결과에 따라 top/middle/side 등급 모델 자동 선택
```

실서비스에서는 JSON 없이 사과 사진 1장만 입력받습니다. JSON은 학습 준비와 평가 분석에만 사용하고, 최종 추론은 이미지 하나만으로 동작하도록 구성했습니다.

Kaggle 실행 후 `/kaggle/working/models/`에서 아래 파일을 다운로드하면 됩니다.

```text
apple_view_top_middle_side_balanced_resnet18_best.pt
apple_top_grade_resnet18_best.pt
apple_middle_grade_resnet18_best.pt
apple_side_grade_resnet18_best.pt
```

FastAPI 적용 시에는 업로드 이미지를 저장한 뒤 `predict_apple_single(image_path)`를 호출하고, `to_api_response()` 결과를 앱에 반환하면 됩니다.
